# Lección 12.4 — Encoder (BERT) vs. Decoder (GPT) vs. Encoder-Decoder (T5)

> **Módulo 12 · Atención y Transformers** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 70 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 12.2 (máscara causal), Lección 12.3 (bloque Transformer), Lección 11.5 (seq2seq)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Distinguir las **tres familias** de Transformers y decir qué tarea resuelve mejor cada una.
- Explicar la diferencia con **máscaras de atención**: bidireccional, causal y cruzada.
- Describir **cómo se entrena** cada familia (adivinar la siguiente palabra, rellenar huecos, reconstruir fragmentos).
- Abrir **GPT-2**, comprobar que es un decodificador y generar texto **token por token**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (6, 4)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: lector, escritor y traductor

Con los mismos bloques de la Lección 12.3 se arman **tres tipos** de modelo. Piensa en tres oficios:

| Oficio | Qué hace | Cómo mira el texto | Familia | Ejemplo famoso |
|---|---|---|---|---|
| 🧐 **Lector crítico** | Lee un texto **completo** y lo entiende: ¿es positivo?, ¿de qué habla? | Mira **todo**, hacia atrás y hacia adelante | **Encoder** (codificador) | BERT (Google, 2018) |
| ✍️ **Escritor** | Escribe palabra por palabra, continuando lo anterior | Solo mira **hacia atrás** (lo ya escrito) | **Decoder** (decodificador) | GPT-2, GPT-4, Claude, Llama |
| 🌐 **Traductor** | Lee el original **completo** y luego escribe la traducción palabra por palabra | El lector mira todo; el escritor mira lo que ya escribió **y** consulta el original | **Encoder-Decoder** | T5 (Google, 2019), el Transformer original |

Es el seq2seq de la Lección 11.5, pero con atención en lugar de RNN: el **encoder** es el que lee y el **decoder** el que escribe.

### ✋ Ejercicio 1 — ¿Qué familia contratarías?
Para cada tarea, elige **encoder**, **decoder** o **encoder-decoder**:
1. Clasificar correos como spam / no spam.
2. Un asistente que conversa y escribe respuestas largas.
3. Traducir manuales del inglés al español.
4. Buscar, entre 10,000 documentos, los más parecidos a una pregunta.
5. Resumir noticias largas en 3 líneas.
6. Autocompletar código mientras programas.

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Encoder**: hay que entender el correo completo y dar una etiqueta; no hay que generar texto.
2. **Decoder**: generar texto palabra por palabra.
3. **Encoder-decoder** es la elección clásica (leer todo el original y luego escribir). Hoy un decoder grande también traduce muy bien si le das el original como inicio del texto.
4. **Encoder**: convierte cada documento y la pregunta en vectores (*embeddings*, Lección 11.2) y compara. Es la base de los buscadores y de RAG (Lección 13.8).
5. **Encoder-decoder** (clásico) o **decoder** (moderno), por la misma razón que la 3.
6. **Decoder**: continuar lo que ya está escrito.

Fíjate en que los decoders grandes "se comieron" muchas tareas: si sabes **continuar** cualquier texto, puedes resolver casi cualquier problema planteado como texto. Por eso hoy dominan los decoders (los LLM del Módulo 13).
</details>

---
## 2. 📖 El concepto: la diferencia está en la máscara

Las tres familias usan los mismos bloques. Lo que cambia es **a quién puede mirar cada token**, es decir, la **máscara** de atención (Lección 12.2):

- **Encoder → atención bidireccional:** cada token mira a todos.
- **Decoder → atención causal:** cada token mira solo a los anteriores (y a sí mismo).
- **Encoder-decoder** usa las dos, más una tercera: la **atención cruzada**. Las **consultas** salen del decoder (lo que estoy escribiendo) y las **claves y valores** salen del encoder (el original). En la analogía de la biblioteca: el escritor consulta la biblioteca que armó el lector.

In [ ]:
original = ["the", "black", "cat", "sleeps"]             # lo que lee el encoder
traduccion = ["el", "gato", "negro", "duerme"]           # lo que escribe el decoder
n = 4

bidireccional = np.ones((n, n))
causal = np.tril(np.ones((n, n)))                        # triángulo inferior
cruzada = np.ones((n, n))                                # cada palabra escrita puede mirar TODO el original

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, M, titulo, filas, cols in [
        (axes[0], bidireccional, "Encoder: bidireccional", original, original),
        (axes[1], causal, "Decoder: causal", traduccion, traduccion),
        (axes[2], cruzada, "Atención cruzada\n(decoder consulta al encoder)", traduccion, original)]:
    ax.imshow(M, cmap="Greens", vmin=0, vmax=1.3)
    ax.set_xticks(range(n), cols); ax.set_yticks(range(n), filas); ax.set_title(titulo)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, "✔" if M[i, j] else "✖", ha="center", va="center")
axes[0].set_ylabel("quien mira"); axes[1].set_xlabel("a quién puede mirar")
plt.tight_layout(); plt.show()

### 🔮 Ejercicio 2 — Contar lo que se puede ver
Para una frase de **n = 6** tokens:
1. ¿Cuántas casillas permitidas (✔) tiene la atención **bidireccional**?
2. ¿Y la **causal**? (Pista: 1 + 2 + 3 + …)
3. En un encoder-decoder que lee 6 tokens y escribe 4, ¿qué forma tiene la tabla de atención **cruzada**?

✍️ *Mis predicciones:*

In [ ]:
n6 = 6
print("Bidireccional:", int(np.ones((n6, n6)).sum()), "casillas permitidas")
print("Causal       :", int(np.tril(np.ones((n6, n6))).sum()), "casillas permitidas")
print("Cruzada (escribe 4, lee 6):", np.ones((4, n6)).shape)

<details><summary><b>👀 Ver respuesta</b></summary>

1. **36** (6 × 6).
2. **21** (1 + 2 + 3 + 4 + 5 + 6): un poco más de la mitad.
3. **4 × 6**: una fila por cada token que escribe el decoder, una columna por cada token del original. A diferencia de la auto-atención, la tabla cruzada **no** tiene por qué ser cuadrada.
</details>

---
## 3. 📖 Cómo aprende cada familia: tres juegos de entrenamiento

Las tres familias se **preentrenan** con montañas de texto sin etiquetar (aprendizaje **auto-supervisado**, Lección 1.3): el propio texto genera las preguntas y las respuestas. Cambia el "juego":

| Familia | Juego | Ejemplo con *"el gato duerme en el sofá"* |
|---|---|---|
| Decoder (GPT) | **Adivinar la siguiente palabra** | "el gato duerme en el" → **sofá** |
| Encoder (BERT) | **Rellenar huecos** (*masked language modeling*): se tapa ~15 % de las palabras | "el gato [MASK] en el sofá" → **duerme** |
| Encoder-decoder (T5) | **Reconstruir fragmentos borrados** (*span corruption*) | "el gato ‹X› el sofá" → "‹X› duerme en" |

Vamos a fabricar los ejemplos de entrenamiento de cada juego a partir de una frase:

In [ ]:
frase = "el gato negro duerme tranquilo en el sofá de la abuela".split()
rng = np.random.default_rng(3)

print("✍️ DECODER (GPT) — adivinar la siguiente palabra:")
for i in range(1, len(frase)):
    print(f"   {' '.join(frase[:i]):<52} → {frase[i]}")

print("\n🧐 ENCODER (BERT) — rellenar huecos (~15 %):")
tapadas = sorted(rng.choice(len(frase), size=max(1, round(0.15 * len(frase))), replace=False))
print("   entrada:", " ".join("[MASK]" if i in tapadas else w for i, w in enumerate(frase)))
print("   objetivo:", {f"posición {i}": frase[i] for i in tapadas})

print("\n🌐 ENCODER-DECODER (T5) — reconstruir un fragmento:")
ini, fin = 3, 6
print("   entrada :", " ".join(frase[:ini] + ["‹X›"] + frase[fin:]))
print("   objetivo:", "‹X› " + " ".join(frase[ini:fin]))

### 🔮 Ejercicio 3 — ¿Cuántas preguntas por frase?
La frase tiene **11 palabras**. Antes de mirar la salida de arriba (o tápala con la mano):
1. ¿Cuántos ejemplos de "siguiente palabra" saca GPT de esta frase?
2. ¿Cuántas palabras tapa BERT (15 %)?
3. ¿Cuál de los dos juegos aprovecha más cada frase?

✍️ *Mis predicciones:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **10** (todas menos la primera tienen una palabra anterior). ¡Y en el entrenamiento real se calculan **todas a la vez**, en un solo paso, gracias a la máscara causal!
2. **2** palabras (15 % de 11 ≈ 1.65 → 2).
3. **GPT**: aprende de **cada** posición. BERT solo recibe "señal" de las palabras tapadas. A cambio, BERT ve el contexto de los **dos lados**, lo que lo hace muy bueno para *entender*. Es un compromiso.
</details>

---
## 4. 📖 GPT-2 por dentro: un decodificador de verdad

Volvamos a GPT-2 (Lección 12.2). Vamos a comprobar con datos que es un **decoder**.

> ⏳ La primera vez se descarga el modelo (≈500 MB).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained("gpt2")
from transformers.utils import logging
logging.set_verbosity_error(); logging.disable_progress_bar()                   # ocultar avisos técnicos de la librería
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").eval()

cfg = gpt2.config
print(f"Capas: {cfg.n_layer} · cabezas: {cfg.n_head} · d_model: {cfg.n_embd}")
print(f"Contexto máximo: {cfg.n_positions} tokens · vocabulario: {cfg.vocab_size:,} tokens")

### 🔮 Ejercicio 4 — ¿Dónde viven los 124 millones de parámetros?
GPT-2 pequeño tiene ≈124 millones de parámetros. Una parte está en las **tablas de *embeddings*** (un vector de 768 números por cada uno de los 50,257 tokens, más uno por cada una de las 1,024 posiciones). El resto, en los 12 bloques.

¿Qué porcentaje crees que ocupan los *embeddings*: ≈5 %, ≈30 % o ≈70 %?

✍️ *Mi predicción:*

In [ ]:
total = sum(p.numel() for p in gpt2.parameters())
embeddings = gpt2.transformer.wte.weight.numel() + gpt2.transformer.wpe.weight.numel()
por_bloque = sum(p.numel() for p in gpt2.transformer.h[0].parameters())
print(f"Total             : {total:>12,}")
print(f"Embeddings        : {embeddings:>12,}  ({embeddings / total:.0%})")
print(f"Cada bloque       : {por_bloque:>12,}  × {cfg.n_layer} bloques = {por_bloque * cfg.n_layer:,}")

<details><summary><b>👀 Ver respuesta</b></summary>

≈**32 %** (39.4 millones): solo la tabla de tokens tiene 50,257 × 768 ≈ 38.6 millones. Cada bloque tiene ≈7.1 millones (y, como viste en la Lección 12.3, unos 2/3 están en el MLP).

En los modelos gigantes (miles de millones de parámetros) los bloques crecen mucho más que el vocabulario, y los *embeddings* pasan a ser una fracción pequeña. En un modelo "pequeño" como GPT-2, todavía pesan bastante.
</details>

### 🔮 Ejercicio 5 — ¿Es causal de verdad?
Si GPT-2 es un decoder, **todas** sus atenciones (12 capas × 12 cabezas) deberían tener ceros por encima de la diagonal. Vamos a sumar todo lo que hay "en el futuro". ¿Qué número esperas?

✍️ *Mi predicción:*

In [ ]:
entrada = tokenizador("The capital of France is", return_tensors="pt")
with torch.no_grad():
    salida = gpt2(**entrada, output_attentions=True)
A = torch.stack(salida.attentions)[:, 0]           # (capas, cabezas, tokens, tokens)
futuro = torch.triu(torch.ones(A.shape[-1], A.shape[-1], dtype=torch.bool), diagonal=1)
print("Forma de las atenciones:", tuple(A.shape))
print("Peso total puesto en el futuro (144 mapas):", A[..., futuro].sum().item())

<details><summary><b>👀 Ver respuesta</b></summary>

**Exactamente 0.0.** En los 144 mapas de atención, ningún token mira hacia adelante. Es la máscara causal en acción: GPT-2 es un **decodificador** puro. (Un encoder como BERT tendría peso por todas partes.)
</details>

### 📖 Cómo escribe un decoder: un token a la vez
Al final del último bloque, GPT-2 convierte el vector del **último** token en una probabilidad para cada uno de los 50,257 tokens del vocabulario. Generar texto es repetir:
1. Calcular las probabilidades del siguiente token.
2. Elegir uno (aquí: el más probable, lo que se llama *greedy*).
3. Pegarlo al final y volver al paso 1.

### 🔮 Ejercicio 6 — "The capital of France is…"
¿Qué probabilidad crees que le da GPT-2 a **" Paris"** como siguiente token? ¿Será el más probable?

✍️ *Mi predicción:*

In [ ]:
def siguientes(texto, k=5):
    ids = tokenizador(texto, return_tensors="pt").input_ids
    with torch.no_grad():
        probs = gpt2(ids).logits[0, -1].softmax(-1)      # probabilidades del SIGUIENTE token
    p, i = probs.topk(k)
    return [(tokenizador.decode(j), round(v.item(), 3)) for v, j in zip(p, i)]

for token, prob in siguientes("The capital of France is"):     # ✏️ prueba tus propias frases (en inglés)
    print(f"{token!r:<12} {prob:.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

" Paris" sale solo en **5.º lugar, con ≈3 %**. Los más probables son " the" (≈8.5 %), " now", " a"… GPT-2 no está "respondiendo una pregunta": está **continuando un texto**, y en internet "The capital of France is the…" es una continuación muy común ("…is the largest city", "…is the seat of…").

Además, GPT-2 es pequeño y antiguo. Los LLM modernos son mucho más grandes (Lección 12.5) y, además, se ajustan para **seguir instrucciones** (Lección 13.5).
</details>

### ▶️ Generar token por token
Veamos al decoder escribiendo. En cada paso imprimimos sus 3 candidatos favoritos y elegimos el primero.

In [ ]:
texto = "The capital of France is"        # ✏️ prueba otro inicio
ids = tokenizador(texto, return_tensors="pt").input_ids
for paso in range(8):                       # ✏️ cuántos tokens generar
    with torch.no_grad():
        probs = gpt2(ids).logits[0, -1].softmax(-1)
    top_p, top_i = probs.topk(3)
    candidatos = "  ".join(f"{tokenizador.decode(i)!r} {p:.0%}" for p, i in zip(top_p, top_i))
    ids = torch.cat([ids, top_i[:1][None]], dim=1)        # pegar el más probable
    print(f"paso {paso}: {candidatos}")
print("\nTexto final:", tokenizador.decode(ids[0]))

### ✋ Ejercicio 7 — Lee el texto generado
1. ¿Tiene sentido el texto final?
2. ¿Ves algún problema si siguiéramos generando muchos más tokens, eligiendo siempre el más probable? Ejecuta la celda de abajo, que genera 20 tokens con la función `generate` de *transformers*, y compruébalo.

In [ ]:
salida_larga = gpt2.generate(tokenizador("The capital of France is", return_tensors="pt").input_ids,
                             max_new_tokens=20, do_sample=False, pad_token_id=tokenizador.eos_token_id)
print(tokenizador.decode(salida_larga[0]))

<details><summary><b>👀 Ver respuesta</b></summary>

1. Es gramatical pero no responde: *"The capital of France is the capital of the French Republic…"*. Es una continuación "probable", no una respuesta útil.
2. Con 20 tokens aparece la **repetición**: *"…the capital of the French Republic, and the capital of the French Republic is…"*. Elegir siempre el más probable lleva a bucles. Por eso, al generar, se usa **muestreo** con perillas como la **temperatura**, *top-k* y *top-p* (Lección 13.6).
</details>

---
## 5. 📖 ¿Qué pasa si…? Un decoder que puede ver el futuro

En la Lección 12.2 dijimos: si un modelo de lenguaje puede mirar hacia adelante durante el entrenamiento, **hace trampa**. Vamos a comprobarlo de verdad.

Entrenamos **dos** mini-Transformers de caracteres sobre el primer párrafo del *Quijote*, idénticos excepto por **la máscara causal**. Usamos `nn.TransformerEncoderLayer`: es el mismo bloque de la Lección 12.3, ya empaquetado por PyTorch.

In [ ]:
texto = ("en un lugar de la mancha, de cuyo nombre no quiero acordarme, no ha mucho tiempo que vivía un "
         "hidalgo de los de lanza en astillero, adarga antigua, rocín flaco y galgo corredor. una olla de "
         "algo más vaca que carnero, salpicón las más noches, duelos y quebrantos los sábados, lantejas los "
         "viernes, algún palomino de añadidura los domingos, consumían las tres partes de su hacienda. el "
         "resto della concluían sayo de velarte, calzas de velludo para las fiestas, con sus pantuflos de lo "
         "mesmo, y los días de entresemana se honraba con su vellorí de lo más fino. tenía en su casa una ama "
         "que pasaba de los cuarenta, y una sobrina que no llegaba a los veinte, y un mozo de campo y plaza, "
         "que así ensillaba el rocín como tomaba la podadera. frisaba la edad de nuestro hidalgo con los "
         "cincuenta años; era de complexión recia, seco de carnes, enjuto de rostro, gran madrugador y amigo "
         "de la caza.")
letras = sorted(set(texto))
a_numero = {c: i for i, c in enumerate(letras)}
datos = torch.tensor([a_numero[c] for c in texto])
V, CONTEXTO = len(letras), 32

class MiniGPT(nn.Module):
    def __init__(self, causal=True, d_model=64, n_cabezas=4, n_capas=2):
        super().__init__()
        self.causal = causal
        self.embedding = nn.Embedding(V, d_model)
        self.posicion = nn.Embedding(CONTEXTO, d_model)
        self.bloques = nn.ModuleList(
            nn.TransformerEncoderLayer(d_model, n_cabezas, 4 * d_model, dropout=0.0,
                                       batch_first=True, norm_first=True) for _ in range(n_capas))
        self.salida = nn.Linear(d_model, V)

    def forward(self, x):
        T = x.shape[1]
        z = self.embedding(x) + self.posicion(torch.arange(T, device=x.device))
        mascara = nn.Transformer.generate_square_subsequent_mask(T, device=x.device) if self.causal else None
        for bloque in self.bloques:
            z = bloque(z, src_mask=mascara, is_causal=self.causal)
        return self.salida(z)

def lote_texto(n=64):
    inicio = torch.randint(0, len(datos) - CONTEXTO - 1, (n,))
    x = torch.stack([datos[i:i + CONTEXTO] for i in inicio])
    y = torch.stack([datos[i + 1:i + CONTEXTO + 1] for i in inicio])    # el objetivo: el texto corrido 1 letra
    return x.to(device), y.to(device)

def entrenar_gpt(causal, pasos=300):
    torch.manual_seed(0)
    modelo = MiniGPT(causal).to(device)
    opt = torch.optim.Adam(modelo.parameters(), lr=3e-3)
    for paso in range(pasos):
        x, y = lote_texto()
        perdida = nn.functional.cross_entropy(modelo(x).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); perdida.backward(); opt.step()
    return modelo, perdida.item()

@torch.no_grad()
def escribir(modelo, inicio="en un lugar", n=80):
    torch.manual_seed(0)
    x = torch.tensor([[a_numero[c] for c in inicio]], device=device)
    for _ in range(n):
        probs = modelo(x[:, -CONTEXTO:])[0, -1].softmax(-1)
        x = torch.cat([x, torch.multinomial(probs, 1)[None]], dim=1)
    return "".join(letras[i] for i in x[0])

print(f"Texto: {len(texto)} caracteres · {V} caracteres distintos")

### 🔮 Ejercicio 8 — ¿Cuál "aprende" mejor?
Vamos a entrenar el modelo **con** máscara causal y el modelo **sin** máscara (puede ver el futuro).
1. ¿Cuál tendrá la **pérdida de entrenamiento** más baja?
2. ¿Cuál escribirá mejor texto?

✍️ *Mis predicciones:*

In [ ]:
inicio = time.time()
honesto, perdida_honesto = entrenar_gpt(causal=True)
tramposo, perdida_tramposo = entrenar_gpt(causal=False)
print(f"(entrenados en {time.time() - inicio:.0f} s)\n")
print(f"CON máscara  → pérdida de entrenamiento {perdida_honesto:.3f}")
print(f"   escribe: {escribir(honesto)!r}\n")
print(f"SIN máscara  → pérdida de entrenamiento {perdida_tramposo:.3f}")
print(f"   escribe: {escribir(tramposo)!r}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El **tramposo** tiene una pérdida muchísimo más baja (≈0.02 contra ≈0.2): parece "mejor".
2. Pero escribe **basura** ("en un lugarrlgrrouirrurgrrrr…"), mientras que el honesto escribe *"en un lugar de la mancha, de cuyo nombre no quiero acordarme, no hama tiempo…"*.

¿Por qué? Durante el entrenamiento, el objetivo de la posición *i* es la letra *i + 1*, ¡que el tramposo **puede ver**! Solo aprendió a **copiar** la letra de al lado. Al generar, la letra siguiente **todavía no existe**, así que no tiene nada que copiar.

Lección para la vida: **una pérdida de entrenamiento sospechosamente baja es una señal de alerta** (como la fuga de datos de la Lección 2.6). Y una nota: el honesto también repite frases del texto porque solo vio 887 caracteres: memoriza (sobreajuste, Lección 4.4). En el Proyecto del módulo lo entrenarás con el *Quijote* completo.
</details>

---
## 6. 📖 Elegir la familia: la tabla de decisión

| Si tu tarea es… | Familia | Modelos de ejemplo | Por qué |
|---|---|---|---|
| Clasificar textos, extraer entidades, detectar sentimiento | Encoder | BERT, RoBERTa, DeBERTa | Ve todo el texto en ambas direcciones; barato y preciso |
| Buscar documentos parecidos (*embeddings*) | Encoder | Sentence-BERT, E5 | Convierte textos en vectores comparables |
| Generar texto libre, conversar, escribir código | Decoder | GPT, Claude, Llama, Mistral | Escribe token por token |
| Traducir, resumir (entrada → salida bien definidas) | Encoder-decoder | T5, BART, Whisper (audio → texto) | Lee todo el original y escribe consultándolo |
| "Un modelo para todo" | Decoder grande | Los LLM actuales | Casi cualquier tarea se puede plantear como "continúa este texto" |

| Perilla de diseño | Encoder | Decoder | Encoder-decoder |
|---|---|---|---|
| Máscara | Ninguna (bidireccional) | Causal | Bidireccional + causal + cruzada |
| Juego de preentrenamiento | Rellenar huecos | Siguiente token | Reconstruir fragmentos |
| ¿Genera texto con fluidez? | ❌ | ✅ | ✅ |
| ¿Usa contexto de ambos lados? | ✅ | ❌ | ✅ (en la entrada) |

### ✋ Ejercicio 9 — El cliente indeciso
Una startup quiere detectar **reseñas falsas** de productos (sí / no) y tiene 50,000 reseñas etiquetadas. Un ingeniero propone usar un decoder enorme "porque es lo más moderno"; otro, ajustar un encoder pequeño tipo BERT. Da **dos argumentos** a favor de cada opción.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Encoder pequeño (BERT):** (1) la tarea es **clasificar**, no generar, y un encoder ve la reseña completa en ambas direcciones; (2) es **mucho más barato y rápido** de ajustar y de usar con millones de reseñas, y con 50,000 ejemplos etiquetados suele dar excelente precisión.

**Decoder grande:** (1) puede funcionar **sin entrenar nada**, solo con instrucciones o pocos ejemplos en el *prompt* (Lección 13.8); (2) puede además **explicar** en palabras por qué sospecha de una reseña.

Una buena respuesta de ingeniería: empezar con el encoder como **línea base** (Lección 4.7) y comparar costo y precisión antes de pagar por el modelo grande.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma / creencia | Qué está pasando | Qué hacer |
|---|---|---|
| Pérdida de entrenamiento casi 0 en un modelo de lenguaje, pero genera basura | Falta la máscara causal: el modelo copia el futuro (Ejercicio 8) | `is_causal=True` / máscara triangular |
| "Voy a usar BERT para escribir un cuento" | Un encoder no se entrenó para generar de izquierda a derecha | Usar un decoder |
| "GPT no entiende el contexto que viene después" | Cierto **en cada posición**, pero al generar ya leyó todo el *prompt* anterior | Poner la información importante **antes** de la pregunta |
| "Encoder-decoder" = "RNN encoder + RNN decoder" | Es la misma idea del seq2seq (Lección 11.5), pero con bloques Transformer y atención cruzada | — |
| El modelo inglés funciona mal en español | Vocabulario y datos de entrenamiento en otro idioma (lo verás en la práctica integradora) | Usar modelos multilingües |

## 8. 📋 Ficha comparativa

| | Encoder (BERT) | Decoder (GPT) | Encoder-decoder (T5) |
|---|---|---|---|
| **Intuición** | Lector crítico | Escritor | Traductor |
| **Atención** | Bidireccional | Causal | Bidireccional (entrada) + causal (salida) + cruzada |
| **Preentrenamiento** | Rellenar huecos (~15 %) | Siguiente token | Reconstruir fragmentos |
| **Salida típica** | Una etiqueta o un vector por texto/token | Texto nuevo | Texto nuevo condicionado a una entrada |
| **✅ Fortalezas** | Comprensión, clasificación, búsqueda; barato | Generación, versatilidad, escala | Tareas entrada → salida (traducir, resumir, transcribir) |
| **❌ Debilidades** | No genera texto con fluidez | No ve "el futuro" en cada posición; caro a gran escala | Más piezas; menos usado en LLM generales |

## 9. 🧠 Resumen en 5 puntos
1. Las tres familias usan los **mismos bloques**; cambia la **máscara**: bidireccional (encoder), causal (decoder) o ambas más atención **cruzada** (encoder-decoder).
2. Cada familia se preentrena con un juego auto-supervisado: **rellenar huecos** (BERT), **siguiente token** (GPT), **reconstruir fragmentos** (T5).
3. **GPT-2 es un decoder**: sus 144 mapas de atención tienen exactamente 0 en el futuro y genera texto **un token a la vez**.
4. Un decoder sin máscara "aprende" con pérdida casi 0… **copiando** el futuro, y luego no sabe escribir.
5. Encoders para **entender**, decoders para **generar**; los decoders grandes se volvieron "modelos para todo".

---
## 10. 🏋️ Ejercicios finales

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Qué familia usa atención **cruzada**?
- a) Encoder · b) Decoder · c) Encoder-decoder · d) Ninguna

**2.** BERT se preentrena…
- a) adivinando la siguiente palabra · b) rellenando palabras tapadas · c) traduciendo · d) con etiquetas humanas

**3.** En la atención **causal** de una frase de 10 tokens, ¿cuántas casillas están permitidas?
- a) 10 · b) 45 · c) 55 · d) 100

**4.** Quieres un buscador que encuentre preguntas frecuentes parecidas a la del cliente. ¿Qué familia es la más natural?
- a) Encoder · b) Decoder · c) Encoder-decoder

**5.** Un modelo de lenguaje tiene pérdida de entrenamiento 0.01 pero genera letras sin sentido. Lo más probable es que…
- a) le falten capas · b) le falte la máscara causal · c) la tasa de aprendizaje sea baja · d) el vocabulario sea pequeño

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **c)** El decoder consulta al encoder con atención cruzada.
2. **b)** *Masked language modeling*.
3. **c)** 1 + 2 + … + 10 = 55.
4. **a)** Convierte textos en vectores comparables.
5. **b)** Es la trampa del Ejercicio 8.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Al entrenar GPT, una frase de 1,000 tokens produce 999 ejemplos de "siguiente token" **en un solo paso** de cálculo. ¿Por qué es posible hacerlos todos a la vez sin que se mezclen las respuestas? ¿Y por qué un RNN (Lección 11.3) no puede hacer lo mismo?

**B. Caso 💼** Un banco quiere un sistema que lea contratos de 50 páginas y responda preguntas sobre ellos. Propón una arquitectura combinando lo visto (pista: un encoder para buscar los fragmentos relevantes + un decoder para redactar la respuesta). ¿Qué ventaja tiene sobre meter las 50 páginas enteras en un decoder?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Gracias a la **máscara causal**: la posición *i* solo ve los tokens 1…*i*, así que su predicción del token *i + 1* es honesta aunque todas las posiciones se calculen en paralelo. Un RNN necesita procesar el token *i* para obtener la memoria que usa en el *i + 1*: es **secuencial** por naturaleza. Esta capacidad de paralelizar el entrenamiento es una de las razones por las que los Transformers escalan tan bien (Lección 12.5).

**B.** Un encoder convierte cada fragmento del contrato en un vector; ante una pregunta, se buscan los fragmentos más parecidos y solo esos se le dan al decoder para que redacte la respuesta. Es la idea de **RAG** (Lección 13.8). Ventajas: menos tokens (la atención cuesta n², Lección 12.1), más barato y rápido, y se puede **citar** de qué parte del contrato salió la respuesta.
</details>

### 10.3 🧩 Práctica integradora: ¿GPT-2 habla español?

Una editorial latinoamericana quiere usar GPT-2 (gratis y pequeño) para autocompletar textos en español. Te piden un diagnóstico rápido. Lo harás en 5 pasos.

**Paso 1 🔮 — Contar tokens.** GPT-2 aprendió su vocabulario con texto en inglés. Antes de ejecutar: ¿la misma frase en español tendrá **más**, **igual** o **menos** tokens que en inglés?

In [ ]:
frases = {"inglés": "The cat is sleeping on the sofa.",
          "español": "El gato está durmiendo en el sofá."}       # ✏️ prueba tus frases
for idioma, f in frases.items():
    ids = tokenizador(f).input_ids
    print(f"{idioma:<8} {len(ids):>2} tokens: {[tokenizador.decode([i]) for i in ids]}")

<details><summary><b>👀 Ver respuesta</b></summary>

El español usa **14 tokens** contra **8** del inglés (≈1.75 veces más). "gato" se parte en " g" + "ato"; "durmiendo" en tres pedazos; y la "á" acentuada es un token aparte. Para GPT-2, el español es un idioma "caro": cabe menos texto en su contexto de 1,024 tokens y cada palabra requiere más pasos de generación.
</details>

**Paso 2 🔮 — ¿Qué tan "sorprendido" está el modelo?** La **perplejidad** mide la sorpresa del modelo ante un texto: es como el número de opciones entre las que "duda" en cada token (más bajo = el texto le resulta más natural). Predice: ¿en qué idioma será más alta?

In [ ]:
def perplejidad(texto):
    ids = tokenizador(texto, return_tensors="pt").input_ids
    with torch.no_grad():
        perdida = gpt2(ids, labels=ids).loss            # entropía cruzada media (Lección 3.2)
    return perdida.exp().item()

for idioma, f in frases.items():
    print(f"{idioma:<8} perplejidad = {perplejidad(f):.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- Inglés: ≈**40** · Español: ≈**383**.

En español, GPT-2 está casi **10 veces más "perdido"** en cada token. Casi no vio español durante su entrenamiento.
</details>

**Paso 3 ▶️ — La pregunta de la capital, en español.** Usa la función `siguientes` de la Sección 4.

In [ ]:
for token, prob in siguientes("La capital de Francia es"):
    print(f"{token!r:<12} {prob:.1%}")

✍️ *¿Aparece "París"? ¿Qué tipo de tokens propone?*

<details><summary><b>👀 Ver respuesta</b></summary>

"París" **no** aparece entre los 5 primeros. Los candidatos son " la" (≈16 %), "pa" (¡un pedazo de palabra!, ≈10 %), " que", " el", " un": palabras muy frecuentes del español, pero ninguna responde. El modelo apenas reconoce la estructura de la frase.
</details>

**Paso 4 ▶️ — Que escriba en español.**

In [ ]:
ids_es = tokenizador("En un lugar de la Mancha", return_tensors="pt").input_ids
gen_es = gpt2.generate(ids_es, max_new_tokens=25, do_sample=False, pad_token_id=tokenizador.eos_token_id)
print(tokenizador.decode(gen_es[0]))

✍️ *¿Qué opinas del resultado?*

<details><summary><b>👀 Ver respuesta</b></summary>

*"En un lugar de la Mancha de la Mancha de la Mancha de la Mancha…"*: un **bucle** de repetición puro. No es utilizable para una editorial. GPT-2 conoce algunas palabras en español porque internet tiene un poco, pero muy poco, y elegir siempre el token más probable lo encierra en un bucle (Ejercicio 7).
</details>

**Paso 5 ✋ — Tu recomendación.** Escribe 3 líneas para la editorial: ¿usarían GPT-2 tal cual? ¿Qué alternativas propondrías?

<details><summary><b>👀 Ver una respuesta modelo</b></summary>

1. **No** usar GPT-2 tal cual: su vocabulario parte el español en muchos pedazos (≈1.75× tokens) y su perplejidad en español es ≈10× mayor que en inglés.
2. Alternativas: un modelo **multilingüe** o entrenado en español (con un vocabulario que incluya palabras y acentos del español), o **ajustar** (*fine-tuning*, Lección 13.7) un modelo con textos de la editorial.
3. Antes de decidir, comparar candidatos con la misma prueba: tokens por frase y perplejidad en **sus** textos.
</details>

---
## 📚 Para profundizar
- 📄 **Artículos:** Devlin y colegas (2018), *BERT* · Radford y colegas (2019), *Language Models are Unsupervised Multitask Learners* (GPT-2) · Raffel y colegas (2019), *Exploring the Limits of Transfer Learning…* (T5).
- 📘 **Blog:** Jay Alammar, *The Illustrated BERT* y *The Illustrated GPT-2* (inglés).
- 🛠️ **Práctica:** en Colab puedes probar BERT rellenando huecos con `pipeline("fill-mask", model="bert-base-uncased")` de *transformers* (descarga ≈440 MB).
- 🤗 **Hugging Face:** el curso gratuito de *transformers* (en español) explica las tres familias con ejemplos.

---
⬅️ **Anterior:** [12.3 Multi-cabeza, codificación posicional y el bloque Transformer](12.3-multicabeza-posicion-bloque.ipynb) · ➡️ **Siguiente:** [12.5 Por qué los Transformers escalan tan bien: leyes de escalamiento](12.5-leyes-de-escalamiento.ipynb)